In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Settings ---
INPUT_XLSX = "DATA_UFM_combined_TEST_AREA.xlsx" 
SHEET_NAME = "Sheet1"
OUTPUT_XLSX = "DATA_UFM_combined_TEST_AREA_filled.xlsx"

METRICS = [
    "branche_n",
    "branchepct1",
    "branchepct2",
    "branchepct3",
    "branchepct4",
    "branchetxt1",
    "branchetxt2",
    "branchetxt3",
    "branchetxt4",
    "offt_privat",
    "ledighed_nyudd_n",
    "ledighed_nyudd",
    "maanedloen_nyudd_n",
    "maanedloen_nyudd",
    "maanedloenp25_nyudd",
    "maanedloenp75_nyudd",
    "maanedloen_nyudd_aggr",
    "ledighed_10aar_n",
    "ledighed_10aar",
    "maanedloen_10aar_n",
    "maanedloen_10aar",
    "maanedloenp25_10aar",
    "maanedloenp75_10aar",
    "maanedloen_10aar_aggr",
    "arbejdsform_n",
    "arbejdsform_fastedagtimer_pct",
    "arbejdsform_faste_aften_nat_pct",
    "arbejdsform_fleksible_pct",
    "arbejdsform_skiftende_pct",
    "arbejdsform_tilretteselv_pct",
    "arbejdsform_andet_pct",
    "arbejdsform_andet_pct2",
    "arbejdstid_n",
    "arbejdstid_timer",
    "relevans_overens_udd_job_n",
    "relevans_overens_udd_job_ntotal",
    "relevans_overens_udd_job_likert",
    "ruster_til_job_n",
    "ruster_til_job_ntotal",
    "ruster_til_job_likert",
    "udlandet_n",
    "udlandet_dk25_pct",
    "udlandet_dk_pct",
    "udlandet_udlandet_pct",
]

REF_COLS = ["hyppigsteid1", "hyppigsteid2", "hyppigsteid3"]

# --- Helpers ---
def norm_udbud(x):
    """Normalize udbud_id to a clean string key (e.g., 751431.0 -> '751431')."""
    try:
        return str(int(x))
    except Exception:
        return str(x)

def parse_ref(s):
    """
    Parse a hyppigsteid value like 'udb2-kacivfysiknanoteknologi:751431'
    into (artikel_id, udbud_id_str). Returns None if not parseable.
    """
    if pd.isna(s):
        return None
    s = str(s)
    if ":" in s:
        a, u = s.split(":", 1)
        return (a.strip(), norm_udbud(u.strip()))
    return None

# --- Load ---
df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Basic sanity checks
missing_ref_cols = [c for c in REF_COLS if c not in df.columns]
if missing_ref_cols:
    raise ValueError(f"Missing expected reference columns: {missing_ref_cols}")

present_metrics = [m for m in METRICS if m in df.columns]
if not present_metrics:
    raise ValueError("None of the specified metrics are present in the file.")

# Normalize keys
df["artikel_id"] = df["artikel_id"].astype(str)
df["udbud_id_str"] = df["udbud_id"].apply(norm_udbud)

# Build fast lookup by (artikel_id, udbud_id_str)
df_indexed = df.set_index(["artikel_id", "udbud_id_str"])

# Pre-parse references
ref_tuples_per_row = []
for _, row in df.iterrows():
    tuples = [parse_ref(row[c]) for c in REF_COLS]
    ref_tuples_per_row.append([t for t in tuples if t is not None])

# Fill missing values from referenced rows
df_filled = df.copy()
fill_counts = {m: 0 for m in present_metrics}
source_counts = {m: [] for m in present_metrics}

for i, tuples in enumerate(ref_tuples_per_row):
    if not tuples:
        continue
    existing_refs = []
    for t in tuples:
        if t in df_indexed.index:
            existing_refs.append(df_indexed.loc[t])

    if not existing_refs:
        continue

    ref_df = pd.DataFrame(existing_refs)
    # Ensure numeric dtype for metrics
    for m in present_metrics:
        if pd.isna(df_filled.at[i, m]):
            avg_val = pd.to_numeric(ref_df[m], errors="coerce").mean(skipna=True)
            if not np.isnan(avg_val):
                df_filled.at[i, m] = avg_val
                fill_counts[m] += 1
                source_counts[m].append(ref_df[m].notna().sum())

# Save
out_path = Path(OUTPUT_XLSX)
with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    df_filled.to_excel(writer, index=False, sheet_name=SHEET_NAME)

# Optional: print a tiny report when running as a script
print("Filled cells per metric:", fill_counts)
print(f"Saved: {out_path.resolve()}")


Filled cells per metric: {'branche_n': 446, 'branchepct1': 446, 'branchepct2': 446, 'branchepct3': 446, 'branchepct4': 446, 'branchetxt1': 0, 'branchetxt2': 0, 'branchetxt3': 0, 'branchetxt4': 0, 'offt_privat': 446, 'ledighed_nyudd_n': 450, 'ledighed_nyudd': 450, 'maanedloen_nyudd_n': 450, 'maanedloen_nyudd': 450, 'maanedloenp25_nyudd': 450, 'maanedloenp75_nyudd': 450, 'maanedloen_nyudd_aggr': 0, 'ledighed_10aar_n': 427, 'ledighed_10aar': 427, 'maanedloen_10aar_n': 425, 'maanedloen_10aar': 425, 'maanedloenp25_10aar': 425, 'maanedloenp75_10aar': 425, 'maanedloen_10aar_aggr': 0, 'arbejdsform_n': 418, 'arbejdsform_fastedagtimer_pct': 418, 'arbejdsform_faste_aften_nat_pct': 418, 'arbejdsform_fleksible_pct': 418, 'arbejdsform_skiftende_pct': 418, 'arbejdsform_tilretteselv_pct': 418, 'arbejdsform_andet_pct': 418, 'arbejdsform_andet_pct2': 418, 'arbejdstid_n': 418, 'arbejdstid_timer': 418, 'relevans_overens_udd_job_n': 418, 'relevans_overens_udd_job_ntotal': 418, 'relevans_overens_udd_job_lik

NOTE that each variable should reach around 450 the script also struggles with strings as can be seen in the branchetxt1 and so on but that variable might not be that usefull 